# 수집한 뉴스 기사에서 중복 기사 제거 및 뉴스 기사 STM 생성
- 키워드 별 뉴스 기사 url 기준으로 중복 제거
```
article = {
                "id"          : article_id,
                "memory_type" : "stm",
                "session_id"  : session_id,
                "title"       : item.get("title", ""),
                "link"        : item.get("originallink", ""),
                "description" : item.get("description", ""),
                "pubDate"     : pub_date_str,
                "turn_index"  : turn_index,
                "author"      : AUTHOR,
                "category"    : CATEGORY,
            }
```

- 동일 일자 기사를 2시간 간격으로 나눔

In [1]:
from glob import glob
import json

In [3]:
keywords = ['환율','중동 전쟁','유류세','국제유가']

In [3]:
for keyword in keywords:
    json_files_path = glob(f'./demo\\naver_news_{keyword}_*.json')
    input_files = json_files_path
    seen = set()
    merged_items = []
    
    for filename in input_files:
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        for item in data.get("items", []):
            key = item.get("originallink")
            if key and key not in seen:
                seen.add(key)
                merged_items.append(item)
    
    result = {
        "total_collected": len(merged_items),
        "items": merged_items
    }
    with open(f"./demo\\merged_{keyword}.json", "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"병합 완료: {len(merged_items)}개 기사 저장됨")

병합 완료: 2676개 기사 저장됨
병합 완료: 2852개 기사 저장됨
병합 완료: 1054개 기사 저장됨
병합 완료: 2291개 기사 저장됨


In [1]:
import json
import uuid
import os
from email.utils import parsedate_to_datetime
from collections import defaultdict

In [4]:
for keyword in keywords:
    json_files_path = f'./demo\\merged_{keyword}.json'
    print(json_files_path)

./demo\merged_환율.json
./demo\merged_중동 전쟁.json
./demo\merged_유류세.json
./demo\merged_국제유가.json


In [5]:
INPUT_PATH  = "./demo\merged_환율.json"
OUTPUT_DIR  = "./demo\output_news"          # 결과 파일이 저장될 폴더
AUTHOR      = "국민일보"
CATEGORY    = "경제"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [7]:
# ─────────────────────────────────────────
# 설정
# ─────────────────────────────────────────


OUTPUT_DIR  = "output_news"          # 결과 파일이 저장될 폴더
os.makedirs(OUTPUT_DIR, exist_ok=True)


for keyword in keywords:
    json_files_path = f'./demo\\merged_{keyword}.json'

    INPUT_PATH  = json_files_path
    AUTHOR      = "국민일보"
    CATEGORY    = "경제"


    # ─────────────────────────────────────────
    # 유틸
    # ─────────────────────────────────────────
    def short_id() -> str:
        """uuid4 앞 8자리만 사용"""
        return str(uuid.uuid4())[:8]

    def get_slot_key(pub_dt) -> str:
        """
        pubDate → 2시간 슬롯 키 반환
        예) 2026-05-28 03:45 → "20260528_02"  (02시~04시 슬롯)
        """
        slot_hour = (pub_dt.hour // 2) * 2          # 0,2,4,...,22
        return pub_dt.strftime(f"%Y%m%d_{slot_hour:02d}")

    def slot_label(slot_key: str) -> str:
        """
        슬롯 키 → 사람이 읽기 쉬운 파일명용 레이블
        예) "20260528_02" → "20260528_0200-0400"
        """
        date_part, h = slot_key.split("_")
        start_h = int(h)
        end_h   = start_h + 2
        return f"{date_part}_{start_h:02d}00-{end_h:02d}00"

    # ─────────────────────────────────────────
    # 1단계: merged.json 로드 & 2시간 슬롯 분류
    # ─────────────────────────────────────────
    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    slots: dict[str, list] = defaultdict(list)

    skipped = 0
    for item in data.get("items", []):
        try:
            pub_dt = parsedate_to_datetime(item["pubDate"])
        except Exception:
            skipped += 1
            continue
        slots[get_slot_key(pub_dt)].append((pub_dt, item))

    print(f"총 기사 수 : {sum(len(v) for v in slots.values())}개  (파싱 실패: {skipped}개)")
    print(f"슬롯 수    : {len(slots)}개\n")

    # ─────────────────────────────────────────
    # 2단계: 슬롯별로 정렬 → 스키마 변환 → 저장
    # ─────────────────────────────────────────
    for slot_key in sorted(slots.keys()):
        items_in_slot = slots[slot_key]

        # pubDate 오름차순 정렬
        items_in_slot.sort(key=lambda x: x[0])

        session_id = short_id()   # 슬롯(파일)마다 고유 session_id
        articles   = []

        for turn_index, (pub_dt, item) in enumerate(items_in_slot):
            pub_date_str = pub_dt.strftime("%Y-%m-%d %H:%M:%S %z")
            article_id   = f"{session_id}_{turn_index}"

            article = {
                "id"          : article_id,
                "memory_type" : "stm",
                "session_id"  : session_id,
                "title"       : item.get("title", ""),
                "link"        : item.get("originallink", ""),
                "description" : item.get("description", ""),
                "pubDate"     : pub_date_str,
                "turn_index"  : turn_index,
                "author"      : AUTHOR,
                "category"    : CATEGORY,
            }
            articles.append(article)

        result = {
            "stm_articles": [
                {
                    "session_id": session_id,
                    "articles"  : articles,
                }
            ]
        }

        label     = slot_label(slot_key)
        out_path  = os.path.join(OUTPUT_DIR, f"news_{keyword}_{label}.json")

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"  [{label}]  {len(articles):>3}개  session_id={session_id}  → {out_path}")

    print(f"\n완료! 총 {len(slots)}개 파일이 '{OUTPUT_DIR}/' 에 저장되었습니다.")

총 기사 수 : 2676개  (파싱 실패: 0개)
슬롯 수    : 57개

  [20260527_1200-1400]    9개  session_id=ef710bb7  → output_news\news_환율_20260527_1200-1400.json
  [20260527_1400-1600]  148개  session_id=00c9940d  → output_news\news_환율_20260527_1400-1600.json
  [20260527_1600-1800]  151개  session_id=b1082214  → output_news\news_환율_20260527_1600-1800.json
  [20260527_1800-2000]   50개  session_id=377293c4  → output_news\news_환율_20260527_1800-2000.json
  [20260527_2000-2200]   14개  session_id=781ffc17  → output_news\news_환율_20260527_2000-2200.json
  [20260527_2200-2400]   12개  session_id=e0a818a6  → output_news\news_환율_20260527_2200-2400.json
  [20260528_0000-0200]   18개  session_id=0fa9930f  → output_news\news_환율_20260528_0000-0200.json
  [20260528_0200-0400]    1개  session_id=6dc52cb6  → output_news\news_환율_20260528_0200-0400.json
  [20260528_0400-0600]   13개  session_id=864a8085  → output_news\news_환율_20260528_0400-0600.json
  [20260528_0600-0800]   64개  session_id=f66ab7e9  → output_news\news_환율_20260528_06

In [ ]:
import json
import uuid
from email.utils import parsedate_to_datetime

INPUT_PATH  = "naver_news_국제유가.json"
OUTPUT_PATH = f"demo_news_{INPUT_PATH}"

def short_id():
    """uuid4 앞 8자리만 사용"""
    return str(uuid.uuid4())[:8]

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

items = sorted(
    data["items"],
    key=lambda x: parsedate_to_datetime(x["pubDate"])
)

session_id = short_id()  # ex) "550e8400"

articles = []
for turn_index, item in enumerate(items):
    pub_date = parsedate_to_datetime(item["pubDate"])
    pub_date_str = pub_date.strftime("%Y-%m-%d %H:%M:%S %z")

    article_id = f"{session_id}_{short_id()}"  # ex) "550e8400_f47ac10b"

    article = {
        "id": article_id,
        "memory_type": "stm",
        "session_id": session_id,
        "title": item.get("title", ""),
        "link": item.get("originallink", ""),
        # "description": item.get("description", ""),
        "pubDate": pub_date_str,
        "turn_index": turn_index,
        "author": "국민일보",
        "category": "경제"
    }
    articles.append(article)

result = {
    "stm_articles": [
        {
            "session_id": session_id,
            "articles": articles
        }
    ]
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"session_id : {session_id}")
print(f"총 기사 수 : {len(articles)}개")
print(f"저장 완료  : {OUTPUT_PATH}")